# Intuition test: NLP

Questions come first. For each one, write your prediction in the cell marked `Prediction`, then run the code cell below it. Answers are at the end, after every question. Do not scroll down early.

Covers notebooks 01 to 06. Seeded and self-contained.

In [ ]:
import numpy as np
import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
np.set_printoptions(precision=3, suppress=True)

## Q1. TF-IDF score

Three documents:

- Doc 0: `"the cat sat on the mat"`
- Doc 1: `"the dog ran in the park"`
- Doc 2: `"cats and dogs are pets"`

Using `sklearn.TfidfVectorizer` with default settings, which single word has the highest TF-IDF score in doc 0? Is it one of the words that appears only in that document, or is it `"the"`?

Prediction: *(write here)*

In [ ]:
docs = ["the cat sat on the mat", "the dog ran in the park", "cats and dogs are pets"]
vec = TfidfVectorizer()
X = vec.fit_transform(docs)
names = vec.get_feature_names_out()
row = X[0].toarray()[0]
top5 = np.argsort(row)[::-1][:5]
print("Top 5 words in doc 0:")
for i in top5:
    print(f"  {names[i]!r}: {row[i]:.4f}")

## Q2. Word vector similarity on a tiny corpus

Train a `gensim.Word2Vec` model on exactly 5 short sentences (shown in the code cell). `vector_size=10`, `window=2`, `min_count=1`, `seed=42`, `workers=1`. What is the cosine similarity between `"king"` and `"queen"`? Is it positive or negative? Is it higher than the similarity between `"king"` and `"cat"`?

Prediction: *(write here)*

In [ ]:
sentences = [
    ["the", "king", "rules", "the", "kingdom"],
    ["the", "queen", "rules", "the", "kingdom"],
    ["the", "man", "walked", "to", "the", "market"],
    ["the", "woman", "walked", "to", "the", "market"],
    ["the", "cat", "sat", "on", "the", "mat"],
]
w2v = Word2Vec(sentences, vector_size=10, window=2, min_count=1, workers=1, seed=SEED)
sim_kq = w2v.wv.similarity("king", "queen")
sim_kc = w2v.wv.similarity("king", "cat")
print(f"king-queen similarity: {sim_kq:.2f}")
print(f"king-cat similarity:   {sim_kc:.2f}")
print(f"king-queen > king-cat: {sim_kq > sim_kc}")

## Q3. CBOW context window

A CBOW model uses `window_size=2`. The training sentence is:

```
["the", "cat", "sat", "on", "the", "mat"]
```

What are the context words used to predict `"sat"` (index 2)? List them in order.

Prediction: *(write here)*

In [ ]:
tokens = ["the", "cat", "sat", "on", "the", "mat"]
idx, window = 2, 2
ctx = [
    tokens[i]
    for i in range(max(0, idx - window), min(len(tokens), idx + window + 1))
    if i != idx
]
print(f"Target: '{tokens[idx]}'")
print(f"Context words: {ctx}")

## Q4. LDA topic assignment

A document contains only the words `["computer", "algorithm", "data", "code"]`. An LDA model has two topics: Topic 0 is dominated by sports words ("goal", "match", "player"), Topic 1 by technology words ("computer", "data", "model").

Which topic should this document be assigned to, and why? This is a reasoning question. No code is needed.

Prediction: *(write here)*

**Answer.** Topic 1 (technology). LDA infers topic mixtures by matching word distributions. The document's words have high probability under Topic 1's word distribution and near-zero probability under Topic 0's. The inference step finds the topic mix that maximises the probability of observing those words, so the document will be assigned almost entirely to Topic 1.

## Q5. RNN perplexity

A character-level RNN trains for 20 epochs. At the end: training loss = 1.8, test loss = 2.4. Perplexity is `exp(loss)`. What is the training perplexity? The test perplexity? By how much do they differ?

Prediction: *(write here)*

In [ ]:
for name, loss in [("train", 1.8), ("test", 2.4)]:
    print(f"{name}: loss = {loss:.1f}, perplexity = {np.exp(loss):.1f}")
print(f"ratio test/train perplexity: {np.exp(2.4) / np.exp(1.8):.2f}")

## Q6. Scaled dot-product attention

A sequence of 3 tokens has query and key matrices:

```
Q = [[1, 0],
     [0, 1],
     [1, 1]]   d_k = 2

K = [[1, 0],
     [0, 1],
     [0, 0]]
```

Compute `softmax(Q K^T / sqrt(d_k))`. Which row of the resulting attention matrix is most "peaked" (puts the most weight on a single token)?

Prediction: *(write here)*

In [ ]:
Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 0], [0, 1], [0, 0]], dtype=float)
d_k = 2

scores = Q @ K.T / np.sqrt(d_k)
exp_s = np.exp(scores - scores.max(axis=1, keepdims=True))  # numerically stable softmax
attn = exp_s / exp_s.sum(axis=1, keepdims=True)

print("Attention matrix (rows = queries, cols = keys):")
print(attn)
print()
max_weights = attn.max(axis=1)
for i, w in enumerate(max_weights):
    print(f"Row {i}: max weight = {w:.3f}")

## Q7. Sinusoidal positional encoding

With `d_model=4`, the positional encoding formula is:

$$\text{PE}(\text{pos}, 2i) = \sin\!\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right), \quad \text{PE}(\text{pos}, 2i+1) = \cos\!\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)$$

What are `PE(0, :)` and `PE(1, :)`? One of these is trivial to predict from the formula.

Prediction: *(write here)*

In [ ]:
def positional_encoding(max_pos: int, d_model: int) -> np.ndarray:
    pe = np.zeros((max_pos, d_model))
    pos = np.arange(max_pos)[:, None]
    div = np.array([10000 ** (2 * i / d_model) for i in range(d_model // 2)])
    pe[:, 0::2] = np.sin(pos / div)
    pe[:, 1::2] = np.cos(pos / div)
    return pe


enc = positional_encoding(2, 4)
print("PE(0, :):", enc[0])
print("PE(1, :):", enc[1])

## Q8. LSTM vs logistic baseline on movie reviews

On the NLTK `movie_reviews` corpus (2,000 documents, balanced positive/negative), a bag-of-words logistic regression baseline achieves about 83–85% accuracy. An LSTM without pre-trained embeddings trained for 10 epochs achieves about 76–80%. Why might a simple bag-of-words model beat a sequence model here? Name two reasons.

Prediction: *(write here)*

**Answer.** Two reasons:

1. **Small corpus, too many parameters.** 2,000 documents is not enough to train an LSTM's embedding layer and recurrent weights from random initialisation. Logistic regression has far fewer parameters and benefits more from the small-data regime.
2. **Sentiment is largely bag-of-words.** Reviews that say "excellent", "boring", or "awful" are mostly classified correctly from those words alone, without needing to track their position in a 300-token sequence. The sequence structure adds noise rather than signal when the discriminating words are strong and appear in isolation.

---

## Answers

Check each against your prediction.

**Q1.** `"the"` wins, with a score of about 0.605. It appears twice in doc 0 (high TF), and also in doc 1, so its IDF is not zero. Single-document words like `"cat"`, `"sat"`, `"mat"`, and `"on"` each appear once, giving them TF=0.167 versus `"the"`'s TF=0.333. Even with higher IDF, the lower TF keeps them behind. TF-IDF rewards both frequency *and* rarity; `"the"` wins here on frequency.

**Q2.** Both similarities are negative, and king-cat is *higher* (less negative) than king-queen. Five sentences are not enough for Word2Vec to find geometric meaning. The vectors are essentially random, and the similarities are noise. A meaningful king-queen > king-cat result requires at least thousands of sentences. Corpus size matters more than model design for embedding quality.

**Q3.** The context words are `['the', 'cat', 'on', 'the']` — the two tokens to the left and two to the right. Note that the same type `"the"` appears twice; CBOW treats each occurrence as a separate context position.

**Q4.** Topic 1. See answer cell above.

**Q5.** Training perplexity = 6.0, test perplexity = 11.0. The ratio is about 1.82. The model memorises some training patterns and generalises less well, so the test character distribution is roughly 1.8 times more surprising. Perplexity is an exponential of loss, so a 0.6-nats difference in loss becomes a 1.8× gap in perplexity.

**Q6.** Rows 0 and 1 have the same distribution (0.503, 0.248, 0.248) and are more peaked than row 2 (0.401, 0.401, 0.198). Token 0's query `[1,0]` aligns most strongly with key 0 `[1,0]`; token 1's query `[0,1]` aligns with key 1. Token 2's query `[1,1]` aligns equally with keys 0 and 1, so its attention is spread across two positions.

**Q7.** `PE(0, :) = [0, 1, 0, 1]`. At position 0 every sine is `sin(0)=0` and every cosine is `cos(0)=1`. `PE(1, :) ≈ [0.841, 0.540, 0.010, 1.000]`. The first pair oscillates at frequency 1, the second at frequency 0.01 (`10000^(2/4) = 100`). Higher dimensions oscillate slowly, encoding coarse position, while lower dimensions oscillate quickly for fine-grained offsets.

**Q8.** See answer cell above.